In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [23]:
load_dotenv(override=True)

openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

gemini=OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

llama7B = OpenAI(api_key="", base_url="http://localhost:11434/v1")

qwen2=OpenAI(api_key="", base_url="http://localhost:11434/v1")


In [27]:
OPENAI_MODEL = "gpt-5"
GEMINI_MODEL = "gemini-2.5-flash-lite"
LLAMA_MODEL = "codellama:7b"
QWEN_MODEL = "tatkal-v1:latest"

In [7]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '25.1.0',
  'version': 'Darwin Kernel Version 25.1.0: Mon Oct 20 19:32:56 PDT 2025; root:xnu-12377.41.6~2/RELEASE_ARM64_T8132',
  'kernel': '25.1.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin25.1.0'},
 'package_managers': ['xcode-select (CLT)', 'brew'],
 'cpu': {'brand': 'Apple M4',
  'cores_logical': 10,
  'cores_physical': 10,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 17.0.0 (clang-1700.3.19.1)',
   'g++': 'Apple clang version 17.0.0 (clang-1700.3.19.1)',
   'clang': 'Apple clang version 17.0.0 (clang-1700.3.19.1)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [9]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

You’re already set up. Your system has Apple’s C/C++ toolchain installed (Apple clang/g++ 17.0.0 via the Command Line Tools), so you do not need to install anything to compile and run C++.

For fastest typical runtime while keeping standard-conforming behavior, use optimization, CPU tuning for your machine, and ThinLTO:

- compile_command (as a list for subprocess.run):
  ["clang++", "-std=c++20", "-O3", "-mcpu=native", "-flto=thin", "-DNDEBUG", "main.cpp", "-o", "main"]

- run_command:
  ["./main"]

Notes:
- If your code can tolerate more aggressive/less strict floating-point optimizations, you can replace -O3 with -Ofast (and optionally add -ffast-math) for potentially higher speed, at the cost of strict IEEE/standards compliance.
- If you ever do need to (re)install the tools on macOS: run xcode-select --install and follow the prompts.

In [11]:
compile_command =  ["clang++", "-std=c++20", "-O3", "-mcpu=native", "-flto=thin", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [44]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.Note: dont add any sentences only stricktly c++ code
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [45]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)},
    ]


In [14]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [15]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [16]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [17]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [18]:
run_python(pi)

Result: 3.141592656089
Execution Time: 9.879996 seconds


In [19]:
port(openai, OPENAI_MODEL, pi)

In [20]:
def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [21]:
compile_and_run()

Result: 3.141592656089
Execution Time: 0.243069 seconds

Result: 3.141592656089
Execution Time: 0.214943 seconds

Result: 3.141592656089
Execution Time: 0.219435 seconds



In [28]:
port(gemini, GEMINI_MODEL, pi)

In [29]:
compile_and_run()

Result: 3.141592656089
Execution Time: 0.234488 seconds

Result: 3.141592656089
Execution Time: 0.205198 seconds

Result: 3.141592656089
Execution Time: 0.208131 seconds



In [30]:
port(llama7B, LLAMA_MODEL, pi)

In [32]:
compile_and_run()

CalledProcessError: Command '['clang++', '-std=c++20', '-O3', '-mcpu=native', '-flto=thin', '-DNDEBUG', 'main.cpp', '-o', 'main']' returned non-zero exit status 1.

In [33]:
port(qwen2, QWEN_MODEL, pi)

In [34]:
compile_and_run()

Result: -0.858407
Execution Time: 240.003 microseconds

Result: -0.858407
Execution Time: 205.978 microseconds

Result: -0.858407
Execution Time: 206.156 microseconds



In [35]:
import gradio as gr

In [36]:
models=["gpt-5","gemini-2.5-flash-lite","codellama:7b","tatkal-v1:latest"]

clients ={"gpt-5":openai,"gemini-2.5-flash-lite":gemini,"codellama:7b":llama7B,"tatkal-v1:latest":qwen2}

In [46]:
def port(model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = clients[model].chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [47]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [48]:
compile_and_run()

CalledProcessError: Command '['clang++', '-std=c++20', '-O3', '-mcpu=native', '-flto=thin', '-DNDEBUG', 'main.cpp', '-o', 'main']' returned non-zero exit status 1.